# Human and Mouse Pretraining Dataset Summary

This notebook reproduces dataset-scale summary statistics from `human_data_summarize.ipynb` and `mouse_data_summarize.ipynb`, then builds a compact Figure 1 style overview for the pretraining corpus.

The final figure highlights:
- total cells and dataset count per species
- cell composition across spatial platforms
- tissue composition within each species


In [1]:
from pathlib import Path

import anndata as ad
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

sns.set_theme(style="white", context="paper")
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 9.5,
    'figure.titlesize': 14,
    'axes.linewidth': 0.9,
})

BASE_DIR = Path('/fs/ess/PAS1475/yzhong/sf_project/spatial_corpus')
CACHE_DIR = BASE_DIR / 'figure1_cache'
CACHE_DIR.mkdir(exist_ok=True)
LEGACY_CACHE_DIR = BASE_DIR / '_figure1_tmp'
OUTPUT_DIR = BASE_DIR / 'figure1_outputs_07_16'
OUTPUT_DIR.mkdir(exist_ok=True)

SPECIES_COLORS = {
    'human': '#de9785ff',
    'mouse': '#e5cdabff',
}

TISSUE_PALETTES = {
    'human': [
        '#2D8875', '#52AADC', '#7C7979', '#7FABD1',
        '#91ccc0', '#963B79', '#97D0C5', '#B5CE4E',
        '#BD7795', '#C7C1DE', '#C89736', '#D75B4E',
        '#EC6E66', '#EEB6D4', '#F39865', '#F7AC53',
    ],
    'mouse': ['#80D0C3', '#9EAAC4', '#A6DDEA', '#F3A59A', '#F9CDBF'],
}

TOP_TISSUES = {
    'human': None,
    'mouse': None,
}

def pretty_label(text):
    return text.replace('_', ' ').title()


def summarize_species(species, use_cache=True):
    cache_csv = CACHE_DIR / f'{species}_file_level_summary.csv'
    legacy_cache_csv = LEGACY_CACHE_DIR / f'{species}_file_level_summary.csv'
    if use_cache and cache_csv.exists():
        return pd.read_csv(cache_csv)
    if use_cache and legacy_cache_csv.exists():
        df = pd.read_csv(legacy_cache_csv)
        df.to_csv(cache_csv, index=False)
        return df

    files = sorted((BASE_DIR / species).glob('*/*/*.h5ad'))
    rows = []

    for file_path in tqdm(files, desc=f'Reading {species} .h5ad files'):
        tissue = file_path.parent.parent.name
        platform = file_path.parent.name
        adata = ad.read_h5ad(file_path, backed='r')
        rows.append({
            'species': species,
            'tissue': tissue,
            'platform': platform,
            'file': file_path.name,
            'n_cells': int(adata.n_obs),
            'n_genes': int(adata.n_vars),
        })
        try:
            adata.file.close()
        except Exception:
            pass

    df = pd.DataFrame(rows)
    df.to_csv(cache_csv, index=False)
    return df


human_df = summarize_species('human', use_cache=True)
mouse_df = summarize_species('mouse', use_cache=True)
all_df = pd.concat([human_df, mouse_df], ignore_index=True)

print(f'Human: {len(human_df):,} datasets, {human_df.n_cells.sum():,} cells')
print(f'Mouse: {len(mouse_df):,} datasets, {mouse_df.n_cells.sum():,} cells')

Human: 109 datasets, 38,304,299 cells
Mouse: 23 datasets, 5,531,207 cells


/fs/ess/PAS1475/yzhong/sf_project/conda_env/spaGFM_dev/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
species_stats = (
    all_df.groupby('species')
    .agg(
        total_cells=('n_cells', 'sum'),
        n_datasets=('file', 'count'),
        n_tissues=('tissue', 'nunique'),
        n_platforms=('platform', 'nunique'),
    )
    .reset_index()
)
species_stats['total_cells_m'] = species_stats['total_cells'] / 1e6

platform_stats = (
    all_df.groupby(['species', 'platform'])['n_cells']
    .sum()
    .reset_index()
)
platform_stats['cells_m'] = platform_stats['n_cells'] / 1e6

tissue_stats = (
    all_df.groupby(['species', 'tissue'])['n_cells']
    .sum()
    .reset_index()
)
tissue_stats['tissue_fraction'] = tissue_stats.groupby('species')['n_cells'].transform(lambda x: x / x.sum())
tissue_stats['cells_m'] = tissue_stats['n_cells'] / 1e6

display(species_stats)
display(platform_stats.sort_values(['species', 'n_cells'], ascending=[True, False]))
display(tissue_stats.sort_values(['species', 'n_cells'], ascending=[True, False]))

,species,total_cells,n_datasets,n_tissues,n_platforms,total_cells_m
0,human,38304299,109,16,3,38.304299
1,mouse,5531207,23,5,3,5.531207


,species,platform,n_cells,cells_m
2,human,xenium,21162998,21.162998
1,human,merscope,11584083,11.584083
0,human,cosmx,5557218,5.557218
5,mouse,xenium,4726993,4.726993
4,mouse,merscope,717298,0.717298
3,mouse,cosmx,86916,0.086916


,species,tissue,n_cells,tissue_fraction,cells_m
2,human,breast,8741616,0.228215,8.741616
4,human,colon,5737537,0.149788,5.737537
8,human,lung,3806032,0.099363,3.806032
14,human,tonsil,3458926,0.090301,3.458926
9,human,lymph_node,2939851,0.076750,2.939851
7,human,liver,2373073,0.061953,2.373073
15,human,uterine,2343540,0.061182,2.343540
10,human,ovarian,1947540,0.050844,1.947540
12,human,prostate,1908493,0.049825,1.908493
1,human,brain,1518632,0.039647,1.518632


In [3]:
def top_tissue_table(df, species, top_n):
    sub = (
        df[df['species'] == species][['tissue', 'n_cells', 'tissue_fraction', 'cells_m']]
        .sort_values('n_cells', ascending=False)
        .reset_index(drop=True)
    )
    if top_n is not None and len(sub) > top_n:
        keep = sub.iloc[:top_n].copy()
        other = pd.DataFrame({
            'tissue': ['other'],
            'n_cells': [sub.iloc[top_n:]['n_cells'].sum()],
            'tissue_fraction': [sub.iloc[top_n:]['tissue_fraction'].sum()],
            'cells_m': [sub.iloc[top_n:]['cells_m'].sum()],
        })
        sub = pd.concat([keep, other], ignore_index=True)
    sub['label'] = sub['tissue'].map(pretty_label)
    return sub


species_order = ['human', 'mouse']
platform_order = ['xenium', 'merscope', 'cosmx']
species_panel = species_stats.set_index('species').loc[species_order].reset_index()

platform_plot = (
    platform_stats.pivot(index='platform', columns='species', values='n_cells')
    .fillna(0)
    .reindex(index=platform_order, columns=species_order)
)
platform_plot_m = platform_plot / 1e6
platform_totals_m = platform_plot_m.sum(axis=1)

human_tissue_plot = top_tissue_table(tissue_stats, 'human', TOP_TISSUES['human'])
mouse_tissue_plot = top_tissue_table(tissue_stats, 'mouse', TOP_TISSUES['mouse'])

def tissue_palette(species, n_colors):
    if n_colors <= 0:
        return []
    palette = TISSUE_PALETTES[species]
    if n_colors > len(palette):
        raise ValueError(f'{species} needs {n_colors} tissue colors, but only {len(palette)} were provided')
    return palette[:n_colors]

human_colors = tissue_palette('human', len(human_tissue_plot))
mouse_colors = tissue_palette('mouse', len(mouse_tissue_plot))

def plot_cells_by_platform_log(ax):
    ypos = np.arange(len(platform_order))
    human_vals = platform_plot_m['human'].values
    mouse_vals = platform_plot_m['mouse'].values
    bar_h = 0.28
    human_y = ypos - bar_h / 2
    mouse_y = ypos + bar_h / 2

    x_min = 0.05
    ax.barh(human_y, human_vals - x_min, left=x_min, color=SPECIES_COLORS['human'], height=bar_h, label='Human')
    ax.barh(mouse_y, mouse_vals - x_min, left=x_min, color=SPECIES_COLORS['mouse'], height=bar_h, label='Mouse')

    ax.set_xscale('log')
    ax.set_xlim(x_min, 30)
    ax.set_xticks([0.1, 0.3, 1, 3, 10, 30])
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f'{x:g}'))
    ax.xaxis.set_minor_locator(mtick.NullLocator())
    ax.set_xlabel('Cells (millions)', fontsize=10.5, labelpad=9)

    ax.set_yticks(ypos)
    ax.set_yticklabels(['Xenium', 'MERSCOPE', 'CosMx'])
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_linewidth(0.9)
    ax.spines['left'].set_color('#3A3A3A')
    ax.spines['bottom'].set_color('#3A3A3A')
    ax.spines['bottom'].set_linewidth(0.75)
    ax.tick_params(axis='x', length=3, width=0.75, color='#3A3A3A', labelsize=9.5, pad=3)
    ax.tick_params(axis='y', length=0, pad=4)
    ax.grid(axis='x', which='major', color='#E6E6E6', lw=0.6)

    for y, val in zip(human_y, human_vals):
        ax.text(val * 1.07, y, f'{val:.1f}M', va='center', ha='left', fontsize=8.8, color='#333333')
    for y, val in zip(mouse_y, mouse_vals):
        label = f'{val:.2f}M' if val < 0.1 else f'{val:.1f}M'
        ax.text(val * 1.07, y, label, va='center', ha='left', fontsize=8.4, color='#333333')

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[:2],
        labels[:2],
        frameon=False,
        loc='lower left',
        bbox_to_anchor=(0.00, -0.24),
        ncol=2,
        handlelength=0.75,
        columnspacing=0.9,
        handletextpad=0.35,
        borderaxespad=0.0,
    )


def donut_panel(ax, df, colors, species_label, panel_letter=None, legend_x_positions=None, x_shift=0.0):
    ax.set_axis_off()
    if panel_letter is not None:
        ax.text(-0.07, 1.2, panel_letter, transform=ax.transAxes, fontsize=15, fontweight='bold', va='top')
    ax.text(0.20 + x_shift, 0.80, species_label, transform=ax.transAxes, fontsize=9.5, color='#000000', fontweight='regular', ha='center')

    donut_ax = ax.inset_axes([0.01 + x_shift, 0.09, 0.38, 0.66])
    donut_ax.pie(
        df['tissue_fraction'],
        colors=colors,
        startangle=90,
        counterclock=False,
        wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 0.8},
    )
    donut_ax.set_aspect('equal')
    donut_ax.set_axis_off()

    legend_cols = 2 if len(df) > 8 else 1
    col_break = int(np.ceil(len(df) / legend_cols))
    x_positions = legend_x_positions if legend_x_positions is not None else [0.40, 0.77]
    for col in range(legend_cols):
        start = col * col_break
        end = min((col + 1) * col_break, len(df))
        for row_idx, (_, row) in enumerate(df.iloc[start:end].iterrows()):
            y = 0.82 - row_idx * 0.075
            x = x_positions[col] + x_shift
            ax.scatter([x], [y], s=24, color=colors[start + row_idx], transform=ax.transAxes, clip_on=False)
            cell_label = f"{row['cells_m']:.2f}M" if row['cells_m'] < 0.1 else f"{row['cells_m']:.1f}M"
            legend_text = f"{row['label']} ({cell_label})"
            ax.text(x + 0.026, y, legend_text, transform=ax.transAxes, fontsize=7.4, va='center', color='#222222')


# Combined figure: tissue composition on row 1, cells by platform on row 2.
fig_combined = plt.figure(figsize=(9.6, 5.85), facecolor='white', dpi=600)
combined_grid = fig_combined.add_gridspec(2, 1, height_ratios=[0.96, 1.04], hspace=0.15)
tissue_grid = combined_grid[0].subgridspec(1, 2, wspace=0.08)
platform_grid = combined_grid[1].subgridspec(1, 3, width_ratios=[0.16, 0.68, 0.16])
ax_human = fig_combined.add_subplot(tissue_grid[0, 0])
ax_mouse = fig_combined.add_subplot(tissue_grid[0, 1])
ax_platform = fig_combined.add_subplot(platform_grid[0, 1])
fig_combined.subplots_adjust(bottom=0.14, top=0.92, left=0.055, right=0.98)

donut_panel(ax_human, human_tissue_plot, human_colors, f'Human (N = {len(human_df)})', panel_letter=None, legend_x_positions=[0.40, 0.70])
donut_panel(ax_mouse, mouse_tissue_plot, mouse_colors, f'Mouse (N = {len(mouse_df)})', panel_letter=None, x_shift=-0.06)

human_box = ax_human.get_position()
mouse_box = ax_mouse.get_position()
divider_x = (human_box.x1 + mouse_box.x0) / 2
fig_combined.add_artist(plt.Line2D([divider_x, divider_x], [human_box.y0, human_box.y1], transform=fig_combined.transFigure, color='#E6E6E6', lw=0.8))
title_y = max(human_box.y1, mouse_box.y1) + 0.01
fig_combined.text((human_box.x0 + mouse_box.x1) / 2, title_y, 'Tissue composition', ha='center', va='bottom', fontsize=13.2, fontweight='regular')

plot_cells_by_platform_log(ax_platform)
ax_platform.set_title('Cells by platform', fontsize=13.2, fontweight='regular', pad=5)

combined_png_path = OUTPUT_DIR / 'human_mouse_composition_by_platform.png'
combined_pdf_path = OUTPUT_DIR / 'human_mouse_composition_by_platform.pdf'
fig_combined.savefig(combined_png_path, dpi=600, bbox_inches='tight', facecolor='white')
fig_combined.savefig(combined_pdf_path, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Saved combined composition/platform figure to: {combined_png_path}')
print(f'Saved combined composition/platform figure to: {combined_pdf_path}')

Saved combined composition/platform figure to: /fs/ess/PAS1475/yzhong/sf_project/spatial_corpus/figure1_outputs_07_16/human_mouse_composition_by_platform.png
Saved combined composition/platform figure to: /fs/ess/PAS1475/yzhong/sf_project/spatial_corpus/figure1_outputs_07_16/human_mouse_composition_by_platform.pdf


In [4]:
def top_tissue_table(df, species, top_n):
    sub = (
        df[df['species'] == species][['tissue', 'n_cells', 'tissue_fraction', 'cells_m']]
        .sort_values('n_cells', ascending=False)
        .reset_index(drop=True)
    )
    if top_n is not None and len(sub) > top_n:
        keep = sub.iloc[:top_n].copy()
        other = pd.DataFrame({
            'tissue': ['other'],
            'n_cells': [sub.iloc[top_n:]['n_cells'].sum()],
            'tissue_fraction': [sub.iloc[top_n:]['tissue_fraction'].sum()],
            'cells_m': [sub.iloc[top_n:]['cells_m'].sum()],
        })
        sub = pd.concat([keep, other], ignore_index=True)
    sub['label'] = sub['tissue'].map(pretty_label)
    return sub


species_order = ['human', 'mouse']
platform_order = ['xenium', 'merscope', 'cosmx']
species_panel = species_stats.set_index('species').loc[species_order].reset_index()

platform_plot = (
    platform_stats.pivot(index='platform', columns='species', values='n_cells')
    .fillna(0)
    .reindex(index=platform_order, columns=species_order)
)
platform_plot_m = platform_plot / 1e6
platform_totals_m = platform_plot_m.sum(axis=1)

human_tissue_plot = top_tissue_table(tissue_stats, 'human', TOP_TISSUES['human'])
mouse_tissue_plot = top_tissue_table(tissue_stats, 'mouse', TOP_TISSUES['mouse'])

def tissue_palette(species, n_colors):
    if n_colors <= 0:
        return []
    palette = TISSUE_PALETTES[species]
    if n_colors > len(palette):
        raise ValueError(f'{species} needs {n_colors} tissue colors, but only {len(palette)} were provided')
    return palette[:n_colors]

human_colors = tissue_palette('human', len(human_tissue_plot))
mouse_colors = tissue_palette('mouse', len(mouse_tissue_plot))

def plot_cells_by_platform_log(ax):
    ypos = np.arange(len(platform_order))
    human_vals = platform_plot_m['human'].values
    mouse_vals = platform_plot_m['mouse'].values
    bar_h = 0.28
    human_y = ypos - bar_h / 2
    mouse_y = ypos + bar_h / 2

    x_min = 0.05
    ax.barh(human_y, human_vals - x_min, left=x_min, color=SPECIES_COLORS['human'], height=bar_h, label='Human')
    ax.barh(mouse_y, mouse_vals - x_min, left=x_min, color=SPECIES_COLORS['mouse'], height=bar_h, label='Mouse')

    ax.set_xscale('log')
    ax.set_xlim(x_min, 30)
    ax.set_xticks([0.1, 0.3, 1, 3, 10, 30])
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f'{x:g}'))
    ax.xaxis.set_minor_locator(mtick.NullLocator())
    ax.set_xlabel('Cells (millions)', fontsize=10.5, labelpad=9)

    ax.set_yticks(ypos)
    ax.set_yticklabels(['Xenium', 'MERSCOPE', 'CosMx'])
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_linewidth(0.9)
    ax.spines['left'].set_color('#3A3A3A')
    ax.spines['bottom'].set_color('#3A3A3A')
    ax.spines['bottom'].set_linewidth(0.75)
    ax.tick_params(axis='x', length=3, width=0.75, color='#3A3A3A', labelsize=9.5, pad=3)
    ax.tick_params(axis='y', length=0, pad=4)
    ax.grid(axis='x', which='major', color='#E6E6E6', lw=0.6)

    for y, val in zip(human_y, human_vals):
        ax.text(val * 1.07, y, f'{val:.1f}M', va='center', ha='left', fontsize=8.8, color='#333333')
    for y, val in zip(mouse_y, mouse_vals):
        label = f'{val:.2f}M' if val < 0.1 else f'{val:.1f}M'
        ax.text(val * 1.07, y, label, va='center', ha='left', fontsize=8.4, color='#333333')

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[:2],
        labels[:2],
        frameon=False,
        loc='lower left',
        bbox_to_anchor=(0.00, -0.24),
        ncol=2,
        handlelength=0.75,
        columnspacing=0.9,
        handletextpad=0.35,
        borderaxespad=0.0,
    )


def donut_panel(ax, df, colors, species_label, panel_letter=None, legend_x_positions=None, x_shift=0.0):
    ax.set_axis_off()
    if panel_letter is not None:
        ax.text(-0.07, 1.2, panel_letter, transform=ax.transAxes, fontsize=15, fontweight='bold', va='top')
    ax.text(0.20 + x_shift, 0.80, species_label, transform=ax.transAxes, fontsize=9.5, color='#000000', fontweight='regular', ha='center')

    donut_ax = ax.inset_axes([0.01 + x_shift, 0.09, 0.38, 0.66])
    donut_ax.pie(
        df['tissue_fraction'],
        colors=colors,
        startangle=90,
        counterclock=False,
        wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 0.8},
    )
    donut_ax.set_aspect('equal')
    donut_ax.set_axis_off()

    legend_cols = 2 if len(df) > 8 else 1
    col_break = int(np.ceil(len(df) / legend_cols))
    x_positions = legend_x_positions if legend_x_positions is not None else [0.40, 0.77]
    for col in range(legend_cols):
        start = col * col_break
        end = min((col + 1) * col_break, len(df))
        for row_idx, (_, row) in enumerate(df.iloc[start:end].iterrows()):
            y = 0.82 - row_idx * 0.075
            x = x_positions[col] + x_shift
            ax.scatter([x], [y], s=24, color=colors[start + row_idx], transform=ax.transAxes, clip_on=False)
            cell_label = f"{row['cells_m']:.2f}M" if row['cells_m'] < 0.1 else f"{row['cells_m']:.1f}M"
            legend_text = f"{row['label']} ({cell_label})"
            ax.text(x + 0.026, y, legend_text, transform=ax.transAxes, fontsize=7.4, va='center', color='#222222')


# Combined figure: tissue composition on row 1, cells by platform on row 2.
fig_combined = plt.figure(figsize=(9.6, 5.85), facecolor='white', dpi=600)
combined_grid = fig_combined.add_gridspec(2, 1, height_ratios=[0.96, 1.04], hspace=0.15)
tissue_grid = combined_grid[0].subgridspec(1, 2, wspace=0.08)
platform_grid = combined_grid[1].subgridspec(1, 3, width_ratios=[0.16, 0.68, 0.16])
ax_human = fig_combined.add_subplot(tissue_grid[0, 0])
ax_mouse = fig_combined.add_subplot(tissue_grid[0, 1])
ax_platform = fig_combined.add_subplot(platform_grid[0, 1])
fig_combined.subplots_adjust(bottom=0.14, top=0.92, left=0.055, right=0.98)

donut_panel(ax_human, human_tissue_plot, human_colors, f'Human (N = {len(human_df)})', panel_letter=None, legend_x_positions=[0.40, 0.70])
donut_panel(ax_mouse, mouse_tissue_plot, mouse_colors, f'Mouse (N = {len(mouse_df)})', panel_letter=None, x_shift=-0.06)

human_box = ax_human.get_position()
mouse_box = ax_mouse.get_position()
divider_x = (human_box.x1 + mouse_box.x0) / 2
fig_combined.add_artist(plt.Line2D([divider_x, divider_x], [human_box.y0, human_box.y1], transform=fig_combined.transFigure, color='#E6E6E6', lw=0.8))
title_y = max(human_box.y1, mouse_box.y1) + 0.01
fig_combined.text((human_box.x0 + mouse_box.x1) / 2, title_y, 'Tissue composition', ha='center', va='bottom', fontsize=13.2, fontweight='regular')

plot_cells_by_platform_log(ax_platform)
ax_platform.set_title('Cells by platform', fontsize=13.2, fontweight='regular', pad=5)

combined_png_path = OUTPUT_DIR / 'human_mouse_composition_by_platform.png'
combined_pdf_path = OUTPUT_DIR / 'human_mouse_composition_by_platform.pdf'
fig_combined.savefig(combined_png_path, dpi=600, bbox_inches='tight', facecolor='white')
fig_combined.savefig(combined_pdf_path, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Saved combined composition/platform figure to: {combined_png_path}')
print(f'Saved combined composition/platform figure to: {combined_pdf_path}')

Saved combined composition/platform figure to: /fs/ess/PAS1475/yzhong/sf_project/spatial_corpus/figure1_outputs_07_16/human_mouse_composition_by_platform.png
Saved combined composition/platform figure to: /fs/ess/PAS1475/yzhong/sf_project/spatial_corpus/figure1_outputs_07_16/human_mouse_composition_by_platform.pdf
